In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/PSICHIC/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/PSICHIC'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/PSICHIC/script')



In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, precision_recall_curve, auc

from torch_geometric.loader import DataLoader

warnings.filterwarnings("ignore")

# ========= Evaluation functions from the XGBoost workflow =========
def calculate_bedroc(y_true, y_score, bedroc_alpha):
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0

    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]
    x      = (ranks - 0.5) / N

    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    x_max  = (np.arange(1, n_act+1) - 0.5) / N
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    return (rie - rie_min) / (rie_max - rie_min)

def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0, 4.0, 5.0], verbose=True):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have equal lengths")
    
    N = len(y_true)
    n_active = np.sum(y_true == 1)
    
    if n_active == 0:
        if verbose:
            print("Warning: no active compounds; returning EF=0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        n_selected = max(1, int(N * percentage / 100))
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        results[f"EF_{percentage}%"] = ef_value
        
        if verbose:
            print(f"EF_{percentage}%:")
            print(f"  - Selected compounds: {n_selected}/{N} ({percentage}%)")
            print(f"  - Active compounds among selected: {n_active_in_selected}")
            print(f"  - Fraction of active compounds among selected: {n_active_in_selected/n_selected:.4f}")
            print(f"  - Overall fraction of active compounds: {n_active/N:.4f}")
            print(f"  - EF value: {ef_value:.4f}")
            print()
    
    return results

def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))

def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    return float(auc(recall, precision))

def fmt4(x):
    return "NaN" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.4f}"

# ========= Configuration =========
CONFIG = {
    "test_csv": _kp('dataset/KIBA_seen_LUDe_augment_sim08.csv'),
    "ligand_pt": _kp('dataset/preprocessed/PSICHIC/KIBA_seen_LUDe_augment/ligand.pt'),
    "protein_pt": _kp('dataset/preprocessed/PSICHIC/KIBA_seen_LUDe_augment/protein.pt'),
    "trained_model_base": _kp('model/PSICHIC/band_0.10_0.50'),  # Base path
    "batch_size": 16,
    "num_workers": 0,
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "kiba_threshold": 3,  # KIBA < 3 indicates an active compound
}
output_dir = _kp('others/baseline/PSICHIC/results/KIBA_seen_LUDe_augment')
# ========= Import the model =========
# Ensure the model code can be imported
import sys
sys.path.insert(0, _kp('external/PSICHIC'))

from models.net import net
from utils.dataset import ProteinMoleculeDataset

# ========= Load the data dictionary =========
def load_ligand_dict(path: str) -> dict:
    obj = torch.load(path, map_location="cpu")
    if not isinstance(obj, dict):
        raise ValueError("ligand_pt did not load a dict.")
    return obj

def load_protein_dict(path: str) -> dict:
    obj = torch.load(path, map_location="cpu")
    if not isinstance(obj, dict):
        raise ValueError("protein_pt did not load a dict.")
    return obj

print("Loading ligand and protein dictionaries...")
ligand_dict = load_ligand_dict(CONFIG["ligand_pt"])
protein_dict = load_protein_dict(CONFIG["protein_pt"])
print(f"Loaded {len(ligand_dict)} ligands and {len(protein_dict)} proteins")

# ========= Load test data and standardize columns =========
print("\nLoading test data...")
test_df = pd.read_csv(CONFIG["test_csv"])
print(f"Original test data shape: {test_df.shape}")
print(f"Original columns: {list(test_df.columns)}")

# Column-name mapping
column_mapping = {
    'SMILES': 'Ligand',
    'Protein_Sequence': 'Protein'
}

# Rename the columns
for old_name, new_name in column_mapping.items():
    if old_name in test_df.columns:
        test_df = test_df.rename(columns={old_name: new_name})
        print(f"Renamed column: {old_name} -> {new_name}")
    else:
        print(f"Warning: Column '{old_name}' not found in dataframe")

print(f"Updated columns: {list(test_df.columns)}")

# Check that the required columns exist
required_columns = ['Ligand', 'Protein', 'KIBA']
missing_columns = [col for col in required_columns if col not in test_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns after renaming: {missing_columns}")

# Create Label from KIBA: KIBA <= 3 is active (1); KIBA > 3 is inactive (0)
test_df['Label'] = (test_df['KIBA'] <= CONFIG["kiba_threshold"]).astype(int)
print(f"\nCreated Label from KIBA <= {CONFIG['kiba_threshold']}")

# Copy Label to classification_label if the dataset requires that column
if 'classification_label' not in test_df.columns:
    test_df['classification_label'] = test_df['Label']
    print("Created classification_label from Label")

print(f"\nKIBA value statistics:")
print(f"  KIBA range: [{test_df['KIBA'].min():.2f}, {test_df['KIBA'].max():.2f}]")
print(f"  KIBA mean: {test_df['KIBA'].mean():.2f}")
print(f"  KIBA median: {test_df['KIBA'].median():.2f}")

print(f"\nLabel distribution:")
print(f"  Active compounds (Label=1, KIBA <= {CONFIG['kiba_threshold']}): {test_df['Label'].sum()} ({test_df['Label'].mean()*100:.2f}%)")
print(f"  Inactive compounds (Label=0, KIBA > {CONFIG['kiba_threshold']}): {(test_df['Label'] == 0).sum()} ({(1-test_df['Label'].mean())*100:.2f}%)")

# Check the uniprot_id column
if "uniprot_id" not in test_df.columns:
    raise ValueError(f"test_csv missing 'uniprot_id'. Available columns: {list(test_df.columns)}")

uniprot_ids = test_df["uniprot_id"].astype(str).values
unique_uniprots = np.unique(uniprot_ids)
print(f"\nTotal unique UniProt IDs: {len(unique_uniprots)}")

# Preview the data to verify its format
print("\nFirst few rows after processing:")
display_cols = ['Ligand', 'Protein', 'KIBA', 'Label', 'uniprot_id']
if 'classification_label' in test_df.columns:
    display_cols.insert(3, 'classification_label')
print(test_df[display_cols].head())

# Show the distribution of KIBA values near the threshold
print(f"\nSamples near threshold (KIBA = {CONFIG['kiba_threshold']}):")
threshold = CONFIG["kiba_threshold"]
near_threshold = test_df[(test_df['KIBA'] >= threshold - 0.5) & (test_df['KIBA'] <= threshold + 0.5)]
print(f"  Samples with KIBA in [{threshold-0.5}, {threshold+0.5}]: {len(near_threshold)}")
if len(near_threshold) > 0:
    print(f"  Labels in this range: {near_threshold['Label'].value_counts().to_dict()}")

# ========= Model-loading functions =========
def build_model_from_saved(trained_model_path: str, device: str):
    degree_path = os.path.join(trained_model_path, "degree.pt")
    config_path = os.path.join(trained_model_path, "config.json")
    param_path  = os.path.join(trained_model_path, "model.pt")

    for p in [degree_path, config_path, param_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")

    degree_dict = torch.load(degree_path, map_location=device)
    mol_deg, prot_deg = degree_dict["ligand_deg"], degree_dict["protein_deg"]

    with open(config_path, "r") as f:
        cfg = json.load(f)

    model = net(
        mol_deg,
        prot_deg,
        mol_in_channels=cfg["params"]["mol_in_channels"],
        prot_in_channels=cfg["params"]["prot_in_channels"],
        prot_evo_channels=cfg["params"]["prot_evo_channels"],
        hidden_channels=cfg["params"]["hidden_channels"],
        pre_layers=cfg["params"]["pre_layers"],
        post_layers=cfg["params"]["post_layers"],
        aggregators=cfg["params"]["aggregators"],
        scalers=cfg["params"]["scalers"],
        total_layer=cfg["params"]["total_layer"],
        K=cfg["params"]["K"],
        t=cfg["params"].get("t", 1),
        heads=cfg["params"]["heads"],
        dropout=cfg["params"]["dropout"],
        dropout_attn_score=cfg["params"]["dropout_attn_score"],
        drop_atom=cfg["params"].get("drop_atom", 0),
        drop_residue=cfg["params"].get("drop_residue", 0),
        dropout_cluster_edge=cfg["params"].get("dropout_cluster_edge", 0),
        gaussian_noise=cfg["params"].get("gaussian_noise", 0),
        regression_head=cfg["tasks"]["regression_task"],
        classification_head=cfg["tasks"]["classification_task"],
        multiclassification_head=cfg["tasks"]["mclassification_task"],
        device=device,
    ).to(device)

    model.reset_parameters()
    state_dict = torch.load(param_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model

# ========= Classification prediction function =========
@torch.no_grad()
def predict_psichic_classification(model, loader, device):
    """
    Return prediction probabilities for the active class.
    The PSICHIC classification output cls_pred contains logits or probabilities.
    """
    preds = []
    
    for data in tqdm(loader, desc="Predicting", leave=False):
        data = data.to(device)

        reg_pred, cls_pred, mcls_pred, sp_loss, o_loss, cl_loss, attn = model(
            mol_x=data.mol_x,
            mol_x_feat=data.mol_x_feat,
            bond_x=data.mol_edge_attr,
            atom_edge_index=data.mol_edge_index,
            clique_x=data.clique_x,
            clique_edge_index=data.clique_edge_index,
            atom2clique_index=data.atom2clique_index,
            residue_x=data.prot_node_aa,
            residue_evo_x=data.prot_node_evo,
            residue_edge_index=data.prot_edge_index,
            residue_edge_weight=data.prot_edge_weight,
            mol_batch=data.mol_x_batch,
            prot_batch=data.prot_node_aa_batch,
            clique_batch=data.clique_x_batch,
        )

        if cls_pred is None:
            raise RuntimeError("cls_pred is None: classification_head might be off?")

        if not torch.is_tensor(cls_pred):
            raise RuntimeError(f"Expected cls_pred Tensor, got {type(cls_pred)}")

        # cls_pred may contain logits of shape [B, 2] or softmax probabilities
        # Apply softmax if the output contains logits
        if cls_pred.dim() == 2 and cls_pred.shape[1] == 2:
            # For binary classification, select the positive (active) class probability
            probs = torch.softmax(cls_pred, dim=1)[:, 1]  # Column 1 contains the active-class probability
        elif cls_pred.dim() == 1:
            # The output already contains probabilities from sigmoid
            probs = cls_pred
        else:
            # Try sigmoid for other output shapes
            probs = torch.sigmoid(cls_pred.view(-1))
        
        probs_np = probs.detach().cpu().numpy()
        preds.extend(probs_np.tolist())

    return np.array(preds)

# ========= Evaluation functions =========
def eval_subset(mask, y_pred_scores, y_true, name="SUBSET", ef_percents=(0.5, 1.0, 2.0)):
    y_sub = y_true[mask]
    s_sub = y_pred_scores[mask]
    
    out = {}
    out["N"] = int(len(y_sub))
    out["Actives"] = int(np.sum(y_sub == 1))
    out["ActiveRatio"] = float(out["Actives"] / out["N"]) if out["N"] > 0 else np.nan
    
    out["ROC_AUC"] = safe_roc_auc(y_sub, s_sub)
    out["PR_AUC"] = safe_pr_auc(y_sub, s_sub)
    out["BEDROC"] = calculate_bedroc(y_sub, s_sub, bedroc_alpha=80.5)
    
    ef_dict = calculate_ef(y_sub, s_sub, percentages=ef_percents, verbose=False)
    out.update(ef_dict)
    
    return out

# ========= Main evaluation loop =========
n_folds = 5

# Store overall metrics for each fold
overall_metrics_all_folds = []

# Store per-target metrics for each fold
per_target_all_folds = {uniprot: [] for uniprot in unique_uniprots}

# ========= Create the dataset once, outside the loop =========
print("Creating dataset and dataloader...")
test_dataset = ProteinMoleculeDataset(
    test_df,
    ligand_dict,
    protein_dict,
    device="cpu",
    cache_transform=True,
)
print(f"Dataset size: {len(test_dataset)}")

# Release the original dictionary after constructing the dataset
del ligand_dict, protein_dict
import gc; gc.collect()
print("Freed ligand/protein dicts from memory")

for fold in range(1, n_folds + 1):
    print(f"\n{'='*80}")
    print(f"EVALUATING FOLD {fold}")
    print("="*80)
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        follow_batch=["mol_x", "prot_node_aa", "clique_x"]
    )

    # Build the model path
    model_path = os.path.join(CONFIG["trained_model_base"], f"train_valid_{fold}")
    
    if not os.path.exists(model_path):
        print(f"Warning: Model path not found: {model_path}")
        continue
    
    print(f"Loading model from: {model_path}")
    
    # Load the model
    try:
        model = build_model_from_saved(model_path, CONFIG["device"])
        print(f"Model loaded successfully on: {CONFIG['device']}")
    except Exception as e:
        print(f"Error loading model for fold {fold}: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    print(f"Dataset size: {len(test_dataset)}, Batches: {len(test_loader)}")
    
    # Predict using the classification output
    try:
        print("Running predictions...")
        y_pred_scores = predict_psichic_classification(model, test_loader, CONFIG["device"])
        y_true = test_df['Label'].values
        
        # Check that lengths match
        if len(y_pred_scores) != len(y_true):
            raise ValueError(f"Prediction length mismatch: {len(y_pred_scores)} vs {len(y_true)}")
        
        print(f"Prediction scores range: [{y_pred_scores.min():.4f}, {y_pred_scores.max():.4f}]")
        print(f"Mean prediction score: {y_pred_scores.mean():.4f}")
        print(f"Prediction scores std: {y_pred_scores.std():.4f}")
        
    except Exception as e:
        print(f"Error during prediction for fold {fold}: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    # ========= 1) Calculate overall metrics =========
    N_all = len(y_true)
    Actives_all = int(np.sum(y_true == 1))
    ActiveRatio_all = Actives_all / N_all
    
    roc_auc_all = safe_roc_auc(y_true, y_pred_scores)
    pr_auc_all = safe_pr_auc(y_true, y_pred_scores)
    bedroc_all = calculate_bedroc(y_true, y_pred_scores, bedroc_alpha=80.5)
    ef_all = calculate_ef(y_true, y_pred_scores, percentages=(0.5, 1.0, 2.0), verbose=False)
    
    overall_metrics_all_folds.append({
        'Fold': fold,
        'N': N_all,
        'Actives': Actives_all,
        'Active_Ratio': ActiveRatio_all,
        'ROC_AUC': roc_auc_all,
        'PR_AUC': pr_auc_all,
        'BEDROC': bedroc_all,
        'EF_0.5%': ef_all.get('EF_0.5%'),
        'EF_1.0%': ef_all.get('EF_1.0%'),
        'EF_2.0%': ef_all.get('EF_2.0%')
    })
    
    print(f"\nFold {fold} Overall Metrics:")
    print(f"  ROC-AUC = {fmt4(roc_auc_all)}")
    print(f"  PR-AUC = {fmt4(pr_auc_all)}")
    print(f"  BEDROC = {fmt4(bedroc_all)}")
    print(f"  EF_0.5% = {fmt4(ef_all.get('EF_0.5%'))}")
    print(f"  EF_1.0% = {fmt4(ef_all.get('EF_1.0%'))}")
    print(f"  EF_2.0% = {fmt4(ef_all.get('EF_2.0%'))}")
    
    # ========= 2) Evaluate each UniProt target =========
    print(f"Evaluating per-target metrics...")
    for uniprot in unique_uniprots:
        mask_t = (uniprot_ids == uniprot)
        n_samples = np.sum(mask_t)
        
        # Evaluate only targets with at least 10 samples
        if n_samples < 10:
            continue
        
        metrics = eval_subset(mask_t, y_pred_scores, y_true, 
                            name=f"TARGET {uniprot}", ef_percents=(0.5, 1.0, 2.0, 5.0, 10.0))
        metrics["Fold"] = fold
        metrics["Target"] = uniprot
        
        per_target_all_folds[uniprot].append(metrics)
    
    del model
    del test_loader
    torch.cuda.empty_cache()
    import gc; gc.collect()

# Check whether any folds were evaluated successfully
if len(overall_metrics_all_folds) == 0:
    print("\n" + "="*80)
    print("ERROR: No folds were successfully evaluated!")
    print("Please check the model paths and ensure models exist.")
    print("="*80)
    exit(1)

# ========= 3) Calculate means and standard deviations across folds =========
print(f"\n{'='*80}")
print(f"CALCULATING AVERAGE AND STD ACROSS {len(overall_metrics_all_folds)} FOLDS")
print("="*80)

# Overall metric means and standard deviations
overall_df = pd.DataFrame(overall_metrics_all_folds)
overall_mean = overall_df.drop('Fold', axis=1).mean()
overall_std = overall_df.drop('Fold', axis=1).std()

print("\nOVERALL METRICS (Mean ± Std across folds):")
print(f"  ROC-AUC = {overall_mean['ROC_AUC']:.4f} ± {overall_std['ROC_AUC']:.4f}")
print(f"  PR-AUC = {overall_mean['PR_AUC']:.4f} ± {overall_std['PR_AUC']:.4f}")
print(f"  BEDROC = {overall_mean['BEDROC']:.4f} ± {overall_std['BEDROC']:.4f}")
print(f"  EF_0.5% = {overall_mean['EF_0.5%']:.4f} ± {overall_std['EF_0.5%']:.4f}")
print(f"  EF_1.0% = {overall_mean['EF_1.0%']:.4f} ± {overall_std['EF_1.0%']:.4f}")
print(f"  EF_2.0% = {overall_mean['EF_2.0%']:.4f} ± {overall_std['EF_2.0%']:.4f}")

# ========= 4) Draw box plots =========
# Mean per-target metrics
per_target_mean_list = []
ef_05_mean_list = []
ef_1_mean_list = []
ef_2_mean_list = []
ef_5_mean_list = []
ef_10_mean_list = []

for uniprot in unique_uniprots:
    if len(per_target_all_folds[uniprot]) == 0:
        continue
    
    target_df = pd.DataFrame(per_target_all_folds[uniprot])
    
    target_mean = {
        'Target': uniprot,
        'N': int(target_df['N'].mean()),
        'Actives': int(target_df['Actives'].mean()),
        'ActiveRatio': target_df['ActiveRatio'].mean(),
        'ROC_AUC_mean': target_df['ROC_AUC'].mean(),
        'ROC_AUC_std': target_df['ROC_AUC'].std(),
        'PR_AUC_mean': target_df['PR_AUC'].mean(),
        'PR_AUC_std': target_df['PR_AUC'].std(),
        'BEDROC_mean': target_df['BEDROC'].mean(),
        'BEDROC_std': target_df['BEDROC'].std(),
        'EF_0.5%_mean': target_df['EF_0.5%'].mean(),  # Use the correct field names
        'EF_0.5%_std': target_df['EF_0.5%'].std(),
        'EF_1.0%_mean': target_df['EF_1.0%'].mean(),  # Use 1.0% rather than 1%
        'EF_1.0%_std': target_df['EF_1.0%'].std(),  
        'EF_2.0%_mean': target_df['EF_2.0%'].mean(),  # Use 2.0% rather than 2%
        'EF_2.0%_std': target_df['EF_2.0%'].std(),      
        'EF_5.0%_mean': target_df['EF_5.0%'].mean(),  # Use 5.0% rather than 5%
        'EF_5.0%_std': target_df['EF_5.0%'].std(),
        'EF_10.0%_mean': target_df['EF_10.0%'].mean(),  # Use 10.0% rather than 10%
        'EF_10.0%_std': target_df['EF_10.0%'].std()
    }
    
    per_target_mean_list.append(target_mean)
    if not np.isnan(target_mean['EF_0.5%_mean']):
        ef_05_mean_list.append(target_mean['EF_0.5%_mean'])
    if not np.isnan(target_mean['EF_1.0%_mean']):
        ef_1_mean_list.append(target_mean['EF_1.0%_mean'])
    if not np.isnan(target_mean['EF_2.0%_mean']):
        ef_2_mean_list.append(target_mean['EF_2.0%_mean'])  
    if not np.isnan(target_mean['EF_5.0%_mean']):
        ef_5_mean_list.append(target_mean['EF_5.0%_mean'])
    if not np.isnan(target_mean['EF_10.0%_mean']):
        ef_10_mean_list.append(target_mean['EF_10.0%_mean'])

# ========= 5) Print statistics =========
print("\n" + "="*60)
print(f"PER-TARGET EF STATISTICS (Mean of {len(overall_metrics_all_folds)} folds)")
print("="*60)

for ef_name, ef_list in [
    ("EF 0.5%", ef_05_mean_list),
    ("EF 1.0%", ef_1_mean_list),  # Use the correct display names
    ("EF 2.0%", ef_2_mean_list),  # Use the correct display names
    ("EF 5.0%", ef_5_mean_list),  # Use the correct display names
    ("EF 10.0%", ef_10_mean_list)  # Use the correct display names
]:
    if len(ef_list) > 0:
        print(f"\n{ef_name}:")
        print(f"  Count: {len(ef_list)}")
        print(f"  Mean: {np.mean(ef_list):.4f}")
        print(f"  Median: {np.median(ef_list):.4f}")
        print(f"  Std: {np.std(ef_list):.4f}")
        print(f"  Min: {np.min(ef_list):.4f}")
        print(f"  Max: {np.max(ef_list):.4f}")
        print(f"  Q1 (25%): {np.percentile(ef_list, 25):.4f}")
        print(f"  Q3 (75%): {np.percentile(ef_list, 75):.4f}")

per_target_mean_df = pd.DataFrame(per_target_mean_list)
print(f"\nProcessed {len(per_target_mean_df)} targets with >= 10 samples")

# ========= 6) Save results =========

os.makedirs(output_dir, exist_ok=True)  # Add this line

# Save overall metrics for all folds
out_overall_all_folds = os.path.join(output_dir, "psichic_eval_overall_all_folds.csv")
overall_df.to_csv(out_overall_all_folds, index=False)
print(f"\nOverall metrics for all folds saved to: {out_overall_all_folds}")

# Save overall metric means and standard deviations
overall_summary = pd.DataFrame({
    'Metric': ['ROC_AUC', 'PR_AUC', 'BEDROC', 'EF_0.5%', 'EF_1.0%', 'EF_2.0%'],
    'Mean': [overall_mean['ROC_AUC'], overall_mean['PR_AUC'], overall_mean['BEDROC'],
             overall_mean['EF_0.5%'], overall_mean['EF_1.0%'], overall_mean['EF_2.0%']],
    'Std': [overall_std['ROC_AUC'], overall_std['PR_AUC'], overall_std['BEDROC'],
            overall_std['EF_0.5%'], overall_std['EF_1.0%'], overall_std['EF_2.0%']]
})
out_overall_summary = os.path.join(output_dir, "psichic_eval_overall_summary.csv")
overall_summary.to_csv(out_overall_summary, index=False)
print(f"Overall metrics summary (mean±std) saved to: {out_overall_summary}")

# Save mean metrics for each target
out_per_target_mean = os.path.join(output_dir, "psichic_eval_per_target_mean.csv")
per_target_mean_df.to_csv(out_per_target_mean, index=False)
print(f"Per-target mean metrics saved to: {out_per_target_mean}")

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)

Loading ligand and protein dictionaries...
Loaded 18968 ligands and 8 proteins

Loading test data...
Original test data shape: (20956, 8)
Original columns: ['chembl_id', 'uniprot_id', 'KIBA', 'SMILES', 'Protein', 'KLIFS', 'Morgan', '_in_spectra']
Renamed column: SMILES -> Ligand
Updated columns: ['chembl_id', 'uniprot_id', 'KIBA', 'Ligand', 'Protein', 'KLIFS', 'Morgan', '_in_spectra']

Created Label from KIBA <= 3
Created classification_label from Label

KIBA value statistics:
  KIBA range: [-1.15, 10.00]
  KIBA mean: 8.76
  KIBA median: 10.00

Label distribution:
  Active compounds (Label=1, KIBA <= 3): 856 (4.08%)
  Inactive compounds (Label=0, KIBA > 3): 20100 (95.92%)

Total unique UniProt IDs: 8

First few rows after processing:
                                              Ligand  \
0  C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...   
1  C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...   
2  C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...   
3  C[S+]([O-])c1ccc(-c2nc(-c3ccc

Prediction scores range: [0.0000, 0.9972]
Mean prediction score: 0.1131
Prediction scores std: 0.2551

Fold 1 Overall Metrics:
  ROC-AUC = 0.5719
  PR-AUC = 0.0605
  BEDROC = 0.1109
  EF_0.5% = 3.7664
  EF_1.0% = 2.6941
  EF_2.0% = 2.2203
Evaluating per-target metrics...

EVALUATING FOLD 2
Loading model from: /home/s1856122/PSICHIC/PSICHIC-main/dataset/SPECTRA_from_merged/band_0.10_0.50/train_valid_split/train_valid_2/result_100_epoch_es/save_model_seed1
Model loaded successfully on: cuda:0
Dataset size: 20956, Batches: 1310
Running predictions...


Prediction scores range: [0.0002, 0.9880]
Mean prediction score: 0.1254
Prediction scores std: 0.1894

Fold 2 Overall Metrics:
  ROC-AUC = 0.5943
  PR-AUC = 0.1226
  BEDROC = 0.3070
  EF_0.5% = 12.0053
  EF_1.0% = 9.3708
  EF_2.0% = 7.0698
Evaluating per-target metrics...

EVALUATING FOLD 3
Loading model from: /home/s1856122/PSICHIC/PSICHIC-main/dataset/SPECTRA_from_merged/band_0.10_0.50/train_valid_split/train_valid_3/result_100_epoch_es/save_model_seed1
Model loaded successfully on: cuda:0
Dataset size: 20956, Batches: 1310
Running predictions...


Prediction scores range: [0.0001, 0.9943]
Mean prediction score: 0.1537
Prediction scores std: 0.2172

Fold 3 Overall Metrics:
  ROC-AUC = 0.6448
  PR-AUC = 0.0666
  BEDROC = 0.1080
  EF_0.5% = 3.2956
  EF_1.0% = 3.1627
  EF_2.0% = 2.5124
Evaluating per-target metrics...

EVALUATING FOLD 4
Loading model from: /home/s1856122/PSICHIC/PSICHIC-main/dataset/SPECTRA_from_merged/band_0.10_0.50/train_valid_split/train_valid_4/result_100_epoch_es/save_model_seed1
Model loaded successfully on: cuda:0
Dataset size: 20956, Batches: 1310
Running predictions...


Prediction scores range: [0.0001, 0.9806]
Mean prediction score: 0.1625
Prediction scores std: 0.2246

Fold 4 Overall Metrics:
  ROC-AUC = 0.6421
  PR-AUC = 0.1025
  BEDROC = 0.2386
  EF_0.5% = 5.6495
  EF_1.0% = 6.0910
  EF_2.0% = 6.0765
Evaluating per-target metrics...

EVALUATING FOLD 5
Loading model from: /home/s1856122/PSICHIC/PSICHIC-main/dataset/SPECTRA_from_merged/band_0.10_0.50/train_valid_split/train_valid_5/result_100_epoch_es/save_model_seed1
Model loaded successfully on: cuda:0
Dataset size: 20956, Batches: 1310
Running predictions...


Prediction scores range: [0.0000, 0.9998]
Mean prediction score: 0.0923
Prediction scores std: 0.1934

Fold 5 Overall Metrics:
  ROC-AUC = 0.5794
  PR-AUC = 0.0666
  BEDROC = 0.1564
  EF_0.5% = 4.0018
  EF_1.0% = 4.9197
  EF_2.0% = 3.6810
Evaluating per-target metrics...

CALCULATING AVERAGE AND STD ACROSS 5 FOLDS

OVERALL METRICS (Mean ± Std across folds):
  ROC-AUC = 0.6065 ± 0.0347
  PR-AUC = 0.0837 ± 0.0273
  BEDROC = 0.1842 ± 0.0866
  EF_0.5% = 5.7437 ± 3.6109
  EF_1.0% = 5.2477 ± 2.6780
  EF_2.0% = 4.3120 ± 2.1640

PER-TARGET EF STATISTICS (Mean of 5 folds)

EF 0.5%:
  Count: 8
  Mean: 5.7450
  Median: 6.7907
  Std: 2.9708
  Min: 0.0000
  Max: 8.8300
  Q1 (25%): 4.7176
  Q3 (75%): 7.8763

EF 1.0%:
  Count: 8
  Mean: 5.6374
  Median: 5.9420
  Std: 2.6557
  Min: 0.3632
  Max: 9.7244
  Q1 (25%): 4.7953
  Q3 (75%): 6.7874

EF 2.0%:
  Count: 8
  Mean: 4.7750
  Median: 5.0717
  Std: 2.2047
  Min: 0.2724
  Max: 8.1037
  Q1 (25%): 4.1492
  Q3 (75%): 5.8356

EF 5.0%:
  Count: 8
  Mean: 3.